CRAZYFLIE
=========

In [1]:
import argparse
import cflib.crtp
from cflib.crtp.radio_link_statistics import RadioLinkStatistics
from cflib.crazyflie.log import LogConfig
from cflib.crazyflie import Crazyflie
from cflib.crazyflie.syncCrazyflie import SyncCrazyflie
from cflib.crazyflie.syncLogger import SyncLogger
from cflib.crazyflie.swarm import CachedCfFactory
from cflib.crazyflie.swarm import Swarm
import cflib.drivers.crazyradio as crazyradio
from cflib.positioning.motion_commander import MotionCommander

from cflib.utils import uri_helper
from cflib.utils.multiranger import Multiranger
from cflib.utils.reset_estimator import reset_estimator

from collections import defaultdict
from dotenv import load_dotenv
import logging
import numpy as np
import sys
from threading import Event
import time

# Data Visualization

In [2]:
# ------------------------------------
# ENVIRONMENT VARIABLES
# ------------------------------------
load_dotenv('')

# URI to the Crazyflie to connect to
URIs = []
for d in range(1,4):
    URIs.append( uri_helper.uri_from_env(env=f'DRONE{str(d)}_URI', default='radio://0/80/2M/E7E7E7E7E7') )
print(URIs)

['radio://0/80/2M/E7E7E7E701', 'radio://0/80/2M/E7E7E7E702', 'radio://0/95/2M/E7E7E7E703']


In [3]:
# ------------------------------------
# STATISTICS
# ------------------------------------
link_quality = 0
def radio_stats_cb(data):
    global link_quality
    link_quality = data
    
def link_error_cb(data):
    raise Exception

In [12]:
# ------------------------------------
# LOGGING
# ------------------------------------
logging.basicConfig(level=logging.ERROR)
print("Python Version", sys.version)
print(f"Crazyradio 2.0 Version: {crazyradio.Crazyradio().version}")

from collections import defaultdict

# Kalman Filter Logging Callback
t_kalman = []
kalman_logs = defaultdict(list)
def log_pos_cb(timestamp, data, logconf):
    global t_kalman
    global kalman_logs
    print('[%d][%s]: %s' % (timestamp, logconf.name, data))
    t_kalman.append(timestamp)
    kalman_logs['kalman.stateX'].append(data['kalman.stateX'])
    kalman_logs['kalman.stateY'].append(data['kalman.stateY'])
    kalman_logs['kalman.stateZ'].append(data['kalman.stateZ'])

# Radio RSSI Logging Callback
t_rssi = []
rssi_logs = []
def log_radio_cb(timestamp, data, logconf):
    pass#print('[%d][%s]: %s' % (timestamp, logconf.name, data))

# Multi-Ranger Logging Callback
t_range = []
range_logs = defaultdict(list)
def log_range_cb(timestamp, data, logconf):
    global t_range
    global range_logs
    print('[%d][%s]: %s' % (timestamp, logconf.name, data))
    t_range.append(timestamp)
    range_logs['range.front'].append(data['range.front'])
    range_logs['range.right'].append(data['range.right'])
    range_logs['range.back'].append(data['range.back'])
    range_logs['range.left'].append(data['range.left'])
    range_logs['range.zrange'].append(data['range.zrange'])

# Battery Logging Callback
battery_level = 0.0
def log_battery_cb(timestamp, data, logconf):
    global battery_level
    battery_level = data['pm.batteryLevel']

sup_info_lkp = { 
    0:("Can be armed", False), 
    1:("Is armed",False), 
    2:("Auto arm mode", False), 
    3:("Ready to fly", False), 
    4:("Is flying", False), 
    5:("Is tumbled", False), 
    6:("Is locked", False), 
    7:("Is crashed", False), 
    8:("HLC enabled", False), 
    9:("HLT Finished", False), 
    10:("HLC disabled", False)
}
def log_supervisor_cb(timestamp, data, logconf):
    global sup_info_lkp
    # Implement something to check all the cases

isFlying = 0
def log_sys_cb(timestamp, data, logconf):
    global isFlying
    isFlying = data['sys.isFlying']

def log_pos_cb2(timestamp, data, logconf):
    print("-" + " POSITION LOG " + "-" * 20)
    for key, value in data.items():
        print(f"{key}: {value:2.3f}",)
    print("\n-" + "--------------" + "-" * 20)

flag_header = False
def log_radio_cb2(timestamp, data, logconf):
    global flag_header
    global link_quality
    global rssi_logs
    
    if not flag_header:
        print("|-" + " RADIO LOG " + "-" * 20)
        print("| ", end='')
        print(f"Link Quality", end=' |\t')
        for key in data.keys():
            print(f"{key}", end=' |\t')
        print("")
        flag_header = True
    print("| ", end='')
    print(f"{link_quality:2.2f}", end=f'{' '*(12-1)}|\t')
    for key, value in data.items():
        rssi_logs.append(value) if key == 'radio.rssi' else None
        print(f"{value}", end=f'{' '*(len(key)-1)}|\t')
    print("")

def log_error_cb(logconf, msg):
    print("Error when logging %s" % logconf.name)

Python Version 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Crazyradio 2.0 Version: 5.4


In [5]:
# CrazyRadio devices currently connected to the computer
print("\n|- Crazyradio Devices ---" + "-" * 10)
dongles = []
for d in crazyradio._find_devices():
    print(f"|- \t{d.manufacturer}, {d.product} (Serial number: {d.serial_number})")
    dongles.append(d)
print("|------------------------" + "-" * 10)


|- Crazyradio Devices -------------
|- 	Bitcraze AB, Crazyradio 2.0 (Serial number: 57D5141CC2A4510C)
|----------------------------------


**Main**

In [6]:
radio = crazyradio.Crazyradio()

# Initialize the low-level drivers
cflib.crtp.init_drivers(enable_serial_driver=False)

# Interface status
interfaces = cflib.crtp.get_interfaces_status()
print(f"Radio Interface: {interfaces['radio']}")

Radio Interface: Crazyradio version 5.4


In [9]:
# Crazyradio Interface scanning
drones = dict() # Dictionary: { 'drone_uri': driverClass_uri}
for uri in URIs:
    drone_addr = uri.split('/')[-1]
    available = cflib.crtp.scan_interfaces(address=int(drone_addr, 16))
    if available:
        #drones.add(available[0][0])
        """
        drones[available[0][0]] = cflib.crtp.get_link_driver(
            uri=available[0][0], 
            #radio_link_statistics_callback=radio_stats_cb,
            link_error_callback=link_error_cb
        )
        """

        drones[available[0][0]] = 1
drones_uri = list(drones.keys())
print("Crazyflie available: ", drones_uri)

Crazyflie available:  ['radio://0/80/2M/E7E7E7E701']


In [10]:
drone_uri = drones_uri[0]
print("Link", drones[drone_uri])
cf_stats = Crazyflie(rw_cache='./cache')

Link 1


**Logging**

In [11]:
def log_configurations():
    logconf = []

    def log_radio_conf():
        log_conf = LogConfig(name="Radio", period_in_ms=500)
        log_conf.add_variable('radio.rssi', 'uint8_t') # Radio Signal Strength Indicator [dBm]
        log_conf.add_variable('radio.isConnected', 'uint8_t') # Indicator if a packet was received from the radio within the last RADIO_ACTIVITY_TIMEOUT_MS
        log_conf.add_variable('radio.numRxBc', 'uint16_t') # Number of broadcast packets received
        log_conf.add_variable('radio.numRxUc', 'uint16_t') # Number of unicast packets received
        return log_conf
    
    def log_pos_conf():
        log_conf = LogConfig(name='Position', period_in_ms=200)
        log_conf.add_variable('kalman.stateX', 'float')
        log_conf.add_variable('kalman.stateY', 'float')
        log_conf.add_variable('kalman.stateZ', 'float')
        return log_conf
    
    def log_range_conf():
        log_conf = LogConfig(name='Range', period_in_ms=200)
        log_type = 'uint16_t' # Distance from the sensor to an obstacle in mm
        log_conf.add_variable('range.front', 'uint16_t')
        log_conf.add_variable('range.back', 'uint16_t')
        log_conf.add_variable('range.left', 'uint16_t')
        log_conf.add_variable('range.right', 'uint16_t')
        log_conf.add_variable('range.zrange', 'uint16_t')
        return log_conf
    
    def log_battery_conf():
        log_conf = LogConfig(name='Battery', period_in_ms=1000)
        battery_type="uint8_t"
        log_conf.add_variable("pm.batteryLevel", battery_type)
        return log_conf
    
    def log_supervisor_conf():
        log_conf = LogConfig(name='Supervisor', period_in_ms=500)
        log_conf.add_variable('supervisor.info', 'uint16_t')
        return log_conf
    
    def log_sys_conf():
        log_conf = LogConfig(name='Sys', period_in_ms=500)
        log_conf.add_variable('sys.isFlying', 'uint8_t')
        return log_conf
        
    def log_imu_conf():
        log_conf = LogConfig(name='Position', period_in_ms=30)
        imu_type="FP16"
        log_conf.add_variable("acc.x", imu_type)
        log_conf.add_variable("acc.y", imu_type)
        log_conf.add_variable("acc.z", imu_type)
        log_conf.add_variable("gyro.x", imu_type)
        log_conf.add_variable("gyro.y", imu_type)
        log_conf.add_variable("gyro.z", imu_type)
        
    logconf.append( log_radio_conf() )
    logconf.append( log_pos_conf() )
    #logconf.append( log_range_conf() )
    logconf.append( log_battery_conf() )
    logconf.append( log_sys_conf() )
    #logconf.append( log_supervisor_conf() )

    return logconf

In [13]:
import asyncio

print("|--- SyncCrazyflie ---|")
scf = SyncCrazyflie(drone_uri, cf=cf_stats)
print("|- 1. Open Link")
scf.open_link()

# ---[ Logs configuration ]---
print("|- 2. Log Configurations ", end=': ')
log_configs = log_configurations()
print([log_conf.name for log_conf in log_configs])

mc = False
multi_ranger = False


# Log configuration to logging framework
for log_conf in log_configs:
    
    scf.cf.log.add_config(log_conf)

    if log_conf.valid:
        print(f"|- 2. Loading {log_conf.name}")
        
        scf.cf.link_statistics.link_quality_updated.add_callback(radio_stats_cb)
        match(log_conf.name):
            case "Radio":
                log_conf.data_received_cb.add_callback(log_radio_cb)
            case "Position":
                log_conf.data_received_cb.add_callback(log_pos_cb)
            #case "Range":
                #log_conf.data_received_cb.add_callback(log_range_cb)
            case "Battery":
                log_conf.data_received_cb.add_callback(log_battery_cb)
            case "Sys":
                log_conf.data_received_cb.add_callback(log_sys_cb)
            case "Supervisor":
                log_conf.data_received_cb.add_callback(log_supervisor_cb)
        
        log_conf.error_cb.add_callback(log_error_cb)
    else:
        print("One or more of the variables in the configuration was not found in log TOC. No logging will be possible.")

# ---[ Parameter configuration ]---
print("|- 3.1 Wait for params...")
scf.wait_for_params()

# Activate mellinger controller
scf.cf.param.set_value('stabilizer.controller', '0')

# Stabilizer - Controller checks - https://www.bitcraze.io/documentation/repository/crazyflie-firmware/master/functional-areas/sensor-to-control/controllers/
def param_controller(_, value_str):
    group, name = _.split(".")
    controller_type = {0:"Auto select", 1:"PID", 2:"Mellinger", 3:"INDI", 4:"Brescianini", 5:"Lee"}
    value = int(value_str)
    print(f"[{group}]: {name} \"{controller_type[value]} ({value})\" is used!")
        
scf.cf.param.add_update_callback(group="stabilizer", name="controller", cb=param_controller)

# Reset estimator
print("|- 3.2 Wait for reset estimator...", end=' ')
reset_estimator(scf) # resets the Kalman filter and makes the Crazyflie wait until it has an accurate position estimate
time.sleep(1)

#scf.cf.link_statistics.link_quality_updated.add_callback(link_quality_cb)
#scf.cf.link_statistics.stop()
for log_conf in log_configs:
    log_conf.start()

|--- SyncCrazyflie ---|
|- 1. Open Link
|- 2. Log Configurations : ['Radio', 'Position', 'Battery', 'Sys']
|- 2. Loading Radio
|- 2. Loading Position
|- 2. Loading Battery
|- 2. Loading Sys
|- 3.1 Wait for params...
|- 3.2 Wait for reset estimator... [stabilizer]: controller "Auto select (0)" is used!
Waiting for estimator to find position...success!


[307200][Position]: {'kalman.stateX': 0.0005919559625908732, 'kalman.stateY': 0.0001370367972413078, 'kalman.stateZ': 0.009520740248262882}
[307400][Position]: {'kalman.stateX': 0.000615500845015049, 'kalman.stateY': -8.407029963564128e-05, 'kalman.stateZ': 0.00957199651747942}
[307600][Position]: {'kalman.stateX': 0.000628557987511158, 'kalman.stateY': 0.00034743096330203116, 'kalman.stateZ': 0.008659380488097668}
[307800][Position]: {'kalman.stateX': 0.0006879377760924399, 'kalman.stateY': -0.00014292524429038167, 'kalman.stateZ': 0.007628711871802807}
[308000][Position]: {'kalman.stateX': 0.0006655682227574289, 'kalman.stateY': 0.0006562545313499868, 'kalman.stateZ': 0.010533234104514122}
[308200][Position]: {'kalman.stateX': 0.0006253472529351711, 'kalman.stateY': -0.00011492226622067392, 'kalman.stateZ': 0.00910139363259077}
[308400][Position]: {'kalman.stateX': 0.00038159938412718475, 'kalman.stateY': 0.0003571590350475162, 'kalman.stateZ': 0.008595377206802368}
[308600][Position

In [18]:
for log_conf in log_configs:
    log_conf.stop()
scf.close_link()

In [ ]:
if not mc:
    print("|- 4. MOTION COMMANDER CONTROL", end='\n|-----------------------------')
    mc = MotionCommander(scf.cf, default_height=0.5)
else:
    pass

|- 4. MOTION COMMANDER CONTROL
|-----------------------------

[309200][Position]: {'kalman.stateX': 0.0008151995134539902, 'kalman.stateY': 0.0001224290463142097, 'kalman.stateZ': 0.007971600629389286}
[309400][Position]: {'kalman.stateX': 0.000771386781707406, 'kalman.stateY': 0.00038706432678736746, 'kalman.stateZ': 0.010056239552795887}
[309600][Position]: {'kalman.stateX': 0.0007935326430015266, 'kalman.stateY': 0.0004123841936234385, 'kalman.stateZ': 0.01004448439925909}
[309800][Position]: {'kalman.stateX': 0.0008557630935683846, 'kalman.stateY': 0.0008754934533499181, 'kalman.stateZ': 0.0075425342656672}
[310000][Position]: {'kalman.stateX': 0.0008399055222980678, 'kalman.stateY': 0.0003955028369091451, 'kalman.stateZ': 0.009698898531496525}
[310200][Position]: {'kalman.stateX': 0.0008244981290772557, 'kalman.stateY': 0.00018294902110937983, 'kalman.stateZ': 0.011263581924140453}
[310400][Position]: {'kalman.stateX': 0.0009675699984654784, 'kalman.stateY': -0.0001697203260846436, 'kalman.stateZ': 0.008624356240034103}
[310600][Position]: {

In [15]:
print("|- 4.1 MC > Take Off")
mc.take_off()

|- 4.1 MC > Take Off
[321000][Position]: {'kalman.stateX': 0.0007583195110782981, 'kalman.stateY': -0.00026845576940104365, 'kalman.stateZ': 0.011253503151237965}
[321200][Position]: {'kalman.stateX': 0.0007730169454589486, 'kalman.stateY': -0.0006063016480766237, 'kalman.stateZ': 0.00895866472274065}
[321400][Position]: {'kalman.stateX': 0.0007481405627913773, 'kalman.stateY': -0.00029964567511342466, 'kalman.stateZ': 0.008219587616622448}
[321600][Position]: {'kalman.stateX': 0.0006925707566551864, 'kalman.stateY': 7.920659118099138e-05, 'kalman.stateZ': 0.010423175059258938}
[321800][Position]: {'kalman.stateX': 0.00045043384307064116, 'kalman.stateY': -0.0005096912500448525, 'kalman.stateZ': 0.010476470924913883}
[322000][Position]: {'kalman.stateX': 0.00044815620640292764, 'kalman.stateY': -2.050639523076825e-06, 'kalman.stateZ': 0.009828653186559677}
[322200][Position]: {'kalman.stateX': 0.00042567128548398614, 'kalman.stateY': -0.0007080873474478722, 'kalman.stateZ': 0.009372374

[325600][Position]: {'kalman.stateX': 0.04188638925552368, 'kalman.stateY': 0.010436615906655788, 'kalman.stateZ': 0.318998247385025}
[325800][Position]: {'kalman.stateX': 0.04370485618710518, 'kalman.stateY': -0.008941074833273888, 'kalman.stateZ': 0.3996967375278473}
[326000][Position]: {'kalman.stateX': 0.01982012763619423, 'kalman.stateY': -0.01892605982720852, 'kalman.stateZ': 0.4692147374153137}
[326200][Position]: {'kalman.stateX': 0.0024682150688022375, 'kalman.stateY': -0.012452014721930027, 'kalman.stateZ': 0.5262430906295776}
[326400][Position]: {'kalman.stateX': 0.005722605623304844, 'kalman.stateY': 0.005686919204890728, 'kalman.stateZ': 0.5558493733406067}
[326600][Position]: {'kalman.stateX': 0.02216843143105507, 'kalman.stateY': 0.018310360610485077, 'kalman.stateZ': 0.5672673583030701}
[326800][Position]: {'kalman.stateX': 0.04437899962067604, 'kalman.stateY': 0.023201629519462585, 'kalman.stateZ': 0.5687273740768433}
[327000][Position]: {'kalman.stateX': 0.05771904438

In [16]:
print("|- 4.2 Multi-Ranger Setup")
multi_ranger = Multiranger(scf, rate_ms=200, zranger=True)
multi_ranger.start()

# 2 - Pre-Configuration: Multi-Ranger Distance, Flow Deck Height
def is_close(range):
    MIN_DISTANCE = 0.3  # m

    if range is None:
        return False
    else:
        return range < MIN_DISTANCE

# async def func():
#     global mc
#     global multi_ranger

keep_flying = True
while keep_flying:
    VELOCITY = 0.3
    velocity_x = 0.0
    velocity_y = 0.0

    if is_close(multi_ranger.front):
        velocity_x -= VELOCITY
    if is_close(multi_ranger.back):
        velocity_x += VELOCITY
    if is_close(multi_ranger.left):
        velocity_y -= VELOCITY
    if is_close(multi_ranger.right):
        velocity_y += VELOCITY

    if is_close(multi_ranger.up):
        keep_flying = False

    mc.start_linear_motion(
        velocity_x_m=velocity_x, 
        velocity_y_m=velocity_y, 
        velocity_z_m=0,
        rate_yaw=0
    )
    time.sleep(0.1)

if keep_flying == False:
    mc.land()

#loop = asyncio.get_event_loop()
#loop.create_task(func())

|- 4.2 Multi-Ranger Setup
[330800][Position]: {'kalman.stateX': 0.07842186093330383, 'kalman.stateY': -0.012599051930010319, 'kalman.stateZ': 0.5484650135040283}
[331000][Position]: {'kalman.stateX': 0.0772458165884018, 'kalman.stateY': -0.014824768528342247, 'kalman.stateZ': 0.5450345873832703}
[331200][Position]: {'kalman.stateX': 0.07792384922504425, 'kalman.stateY': -0.02054566703736782, 'kalman.stateZ': 0.5455513596534729}
[331400][Position]: {'kalman.stateX': 0.08067259937524796, 'kalman.stateY': -0.01852346770465374, 'kalman.stateZ': 0.5442034006118774}
[331600][Position]: {'kalman.stateX': 0.0805792510509491, 'kalman.stateY': -0.028029024600982666, 'kalman.stateZ': 0.5389212965965271}
[331800][Position]: {'kalman.stateX': 0.079493448138237, 'kalman.stateY': -0.02825283817946911, 'kalman.stateZ': 0.5361515879631042}
[332000][Position]: {'kalman.stateX': 0.07625885307788849, 'kalman.stateY': -0.029872937127947807, 'kalman.stateZ': 0.5328378081321716}
[332200][Position]: {'kalman.

[352800][Position]: {'kalman.stateX': 0.20285093784332275, 'kalman.stateY': -0.2692081034183502, 'kalman.stateZ': 0.05658234283328056}
[353000][Position]: {'kalman.stateX': 0.1784413605928421, 'kalman.stateY': -0.28794917464256287, 'kalman.stateZ': 0.00401597935706377}
[353200][Position]: {'kalman.stateX': 0.19824017584323883, 'kalman.stateY': -0.31086066365242004, 'kalman.stateZ': 0.01088462769985199}
[353400][Position]: {'kalman.stateX': 0.19816891849040985, 'kalman.stateY': -0.31133899092674255, 'kalman.stateZ': 0.010223641991615295}
[353600][Position]: {'kalman.stateX': 0.19384068250656128, 'kalman.stateY': -0.3082127571105957, 'kalman.stateZ': 0.007655041757971048}
[353800][Position]: {'kalman.stateX': 0.1900034099817276, 'kalman.stateY': -0.30552342534065247, 'kalman.stateZ': 0.011709739454090595}
[354000][Position]: {'kalman.stateX': 0.1880563348531723, 'kalman.stateY': -0.3042745292186737, 'kalman.stateZ': 0.010151699185371399}
[354200][Position]: {'kalman.stateX': 0.1855874657

In [ ]:
mc.land()

In [ ]:
if not isFlying:
    print("|- 4.1 MC > Take Off")
    mc.take_off()
else:
    print("SONO QUI")
    pass

if not multi_ranger:
    print("|- 4.2 Multi-Ranger Setup")
    multi_ranger = Multiranger(scf, rate_ms=200, zranger=True)

    # 2 - Pre-Configuration: Multi-Ranger Distance, Flow Deck Height
    def is_close(range):
        MIN_DISTANCE = 0.2  # m

        if range is None:
            return False
        else:
            return range < MIN_DISTANCE

    print("Eseguo il While")
    keep_flying = True
    while keep_flying:
        VELOCITY = 0.3
        velocity_x = 0.0
        velocity_y = 0.0

        if is_close(multi_ranger.front):
            velocity_x -= VELOCITY
        if is_close(multi_ranger.back):
            velocity_x += VELOCITY

        if is_close(multi_ranger.left):
            velocity_y -= VELOCITY
        if is_close(multi_ranger.right):
            velocity_y += VELOCITY

        if is_close(multi_ranger.up):
            keep_flying = False

        mc.start_linear_motion(
            velocity_x_m=velocity_x, 
            velocity_y_m=velocity_y, 
            velocity_z_m=0,
            rate_yaw=0
        )
        time.sleep(0.1)

    mc.land()
else:
    print("SONO QUI 2")
    pass


In [ ]:
print("|- 4.2 Multi-Ranger Setup")
multi_ranger = Multiranger(scf, rate_ms=200, zranger=True)

# 2 - Pre-Configuration: Multi-Ranger Distance, Flow Deck Height
def is_close(range):
    MIN_DISTANCE = 0.2  # m

    if range is None:
        return False
    else:
        return range < MIN_DISTANCE

keep_flying = True
while keep_flying:
    VELOCITY = 0.3
    velocity_x = 0.0
    velocity_y = 0.0

    if is_close(multi_ranger.front):
        velocity_x -= VELOCITY
    if is_close(multi_ranger.back):
        velocity_x += VELOCITY

    if is_close(multi_ranger.left):
        velocity_y -= VELOCITY
    if is_close(multi_ranger.right):
        velocity_y += VELOCITY

    if is_close(multi_ranger.up):
        keep_flying = False

    mc.start_linear_motion(
        velocity_x_m=velocity_x, 
        velocity_y_m=velocity_y, 
        velocity_z_m=0,
        rate_yaw=0
    )
    time.sleep(0.1)

mc.land()


In [ ]:
keep_flying = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('_mpl-gallery')

# plot
t1 = range(0,len(kalman_logs['kalman.stateX']))
t2 = range(0,len(kalman_logs['kalman.stateY']))
t3 = range(0,len(kalman_logs['kalman.stateZ']))
tt = [t3, t2, t1]
states = [kalman_logs['kalman.stateZ'], kalman_logs['kalman.stateY'], kalman_logs['kalman.stateX']]
colors = ['red', 'green', 'blue']

fig, axs = plt.subplots(3, 1, layout='constrained', figsize=(6, 6))
for nn, ax in enumerate(axs):
    t = tt.pop()
    state = states.pop()
    color = colors.pop()
    ax.plot(t, state, linewidth=2.0, color=color)

plt.show()

In [ ]:
# plot
t1 = range(0,len(range_logs['range.left']))
t2 = range(0,len(range_logs['range.right']))
t3 = range(0,len(range_logs['range.front']))
t4 = range(0,len(range_logs['range.back']))
tt = [t4, t3, t2, t1]
states = [
    range_logs['range.left'], 
    range_logs['range.right'], 
    range_logs['range.front'],
    range_logs['range.back']
]
colors = ['orange', 'red', 'green', 'blue']

fig, axs = plt.subplots(4, 1, layout='constrained', figsize=(6, 6))
for nn, ax in enumerate(axs):
    t = tt.pop()
    state = states.pop()
    color = colors.pop()
    ax.plot(t, state, linewidth=2.0, color=color)

plt.show()

In [ ]:
# 2 Hz (200 ms)
#print(t_kalman)
t_kalman.index(t_kalman[-2])

In [ ]:
len(t_kalman)

In [ ]:
import random
import time
from collections import defaultdict
import pprint

def generator_kalman(seed=1.0):
    log = {'kalman.stateX': random.random(), 'kalman.stateY': random.random(), 'kalman.stateZ': random.random()}
    yield log

logs = defaultdict(list)
while True:
    try:
        new_entry = next(generator_kalman())                
        logs['kalman.stateX'].append(new_entry['kalman.stateX'])
        logs['kalman.stateY'].append(new_entry['kalman.stateY'])
        logs['kalman.stateZ'].append(new_entry['kalman.stateZ'])
        time.sleep(0.8)
    except KeyboardInterrupt:
        break

pprint.pp(logs)